# Modelado MLP (2018–2024)

Este notebook forma parte del pipeline de ciencia de datos del proyecto **crash-severity-predictor**.

El objetivo es desarrollar, entrenar y evaluar un modelo **Multilayer Perceptron (MLP)** para la predicción de severidad en hechos de tránsito a partir del dataset procesado durante las fases de EDA y ETL. Esta implementación constituye la tercera iteración dentro del conjunto de modelos candidatos del proyecto.


In [1]:
# -- Importaciones ----------------------------------------------
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, accuracy_score,
                             roc_curve)
import plotly.graph_objects as go
import plotly.express as px
import json, os
import warnings
warnings.filterwarnings('ignore')

print('✓ Librerías cargadas correctamente')

✓ Librerías cargadas correctamente


## 1. Carga de datos

In [2]:
# -- Carga ----------------------------------------------
train = pd.read_parquet('../data/clean/train.parquet')
test  = pd.read_parquet('../data/clean/test.parquet')

FEATURES = ['tipo_eve','tipo_veh','g_hora_5','dia_sem_ocu',
            'sexo_per','edad_quinquenales','mayor_menor','depto_ocu']
TARGET = 'fall_les'

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

# -- Escalar features (requerido por MLP) ----------------------------------------------
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train : {X_train_sc.shape}')
print(f'Test  : {X_test_sc.shape}')
print(f'\n✓ Features escaladas correctamente')

Train : (93120, 8)
Test  : (14389, 8)

✓ Features escaladas correctamente


## 2. Entrenamiento MLP

In [3]:
# -- Entrenamiento ----------------------------------------------
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42,
    verbose=False
)

mlp.fit(X_train_sc, y_train)
print(f'✓ Modelo entrenado correctamente')
print(f'Capas ocultas  : {mlp.hidden_layer_sizes}')
print(f'Iteraciones    : {mlp.n_iter_}')
print(f'Loss final     : {mlp.loss_:.4f}')

✓ Modelo entrenado correctamente
Capas ocultas  : (128, 64, 32)
Iteraciones    : 57
Loss final     : 0.5404


## 3. Evaluación del modelo

In [4]:
# -- Predicciones ----------------------------------------------
y_pred  = mlp.predict(X_test_sc)
y_proba = mlp.predict_proba(X_test_sc)[:, 0]

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='weighted')
auc = roc_auc_score(y_test, y_proba)
cm  = confusion_matrix(y_test, y_pred)

print('=== MLP ===')
print(f'Accuracy : {acc:.4f}')
print(f'F1-Score : {f1:.4f}')
print(f'ROC-AUC  : {auc:.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["Fallecido","Lesionado"])}')
print(f'Matriz de confusión:')
print(cm)

=== MLP ===
Accuracy : 0.6500
F1-Score : 0.6861
ROC-AUC  : 0.3058

              precision    recall  f1-score   support

   Fallecido       0.30      0.64      0.41      2748
   Lesionado       0.88      0.65      0.75     11641

    accuracy                           0.65     14389
   macro avg       0.59      0.65      0.58     14389
weighted avg       0.77      0.65      0.69     14389

Matriz de confusión:
[[1756  992]
 [4044 7597]]


In [5]:
# -- Fix: identificar clase positiva correcta ----------------------------------------------
print(f'Clases del modelo: {mlp.classes_}')
print(f'predict_proba columna 0 = clase {mlp.classes_[0]}')
print(f'predict_proba columna 1 = clase {mlp.classes_[1]}')

# Usar columna correcta para ROC
y_proba = mlp.predict_proba(X_test_sc)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f'\nROC-AUC corregido: {auc:.4f}')

Clases del modelo: [1 2]
predict_proba columna 0 = clase 1
predict_proba columna 1 = clase 2

ROC-AUC corregido: 0.6942


In [6]:
# -- Recalcular métricas completas con proba corregida ----------------------------------------------
fpr, tpr, _ = roc_curve(y_test, y_proba, pos_label=1)

print('=== MLP (corregido) ===')
print(f'Accuracy : {acc:.4f}')
print(f'F1-Score : {f1:.4f}')
print(f'ROC-AUC  : {auc:.4f}')

=== MLP (corregido) ===
Accuracy : 0.6500
F1-Score : 0.6861
ROC-AUC  : 0.6942


### Resultados MLP

| Métrica | Valor |
|---|---|
| Accuracy | 65.00% |
| F1-Score (weighted) | 68.61% |
| ROC-AUC | 69.42% |
| Precision Fallecido | 30% |
| Recall Fallecido | 64% |

57 iteraciones con early stopping , convergencia rápida.

In [8]:
# -- Métricas ----------------------------------------------
fig_metricas = go.Figure(go.Bar(
    x=['Accuracy', 'F1-Score', 'ROC-AUC'],
    y=[acc, f1, auc],
    text=[f'{acc:.4f}', f'{f1:.4f}', f'{auc:.4f}'],
    textposition='auto',
    marker_color=['#8338EC', '#3A86FF', '#FF006E'],
    width=0.4
))
fig_metricas.update_layout(
    title='Métricas de evaluación : MLP',
    yaxis=dict(range=[0, 1], title='Valor'),
    xaxis_title='Métrica',
    height=400,
    template='plotly_white'
)
fig_metricas.show()

In [10]:
# -- Matriz de confusión ----------------------------------------------
fig_cm = px.imshow(
    cm,
    labels=dict(x='Predicción', y='Real', color='Cantidad'),
    x=['Fallecido', 'Lesionado'],
    y=['Fallecido', 'Lesionado'],
    text_auto=True,
    color_continuous_scale='Purples',
    title='Matriz de confusión : MLP'
)
fig_cm.update_layout(height=400, template='plotly_white')
fig_cm.show()

In [15]:
# -- curva ROC  ----------------------------------------------
fpr, tpr, _ = roc_curve(y_test, y_proba, pos_label=2)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f'MLP (AUC = {auc:.4f})',
    line=dict(color='#8338EC', width=2.5)
))
fig_roc.add_trace(go.Scatter(
    x=[0,1], y=[0,1],
    mode='lines',
    name='Baseline (AUC = 0.5)',
    line=dict(color='gray', width=1.5, dash='dash')
))
fig_roc.update_layout(
    title='Curva ROC : MLP',
    xaxis_title='Tasa de Falsos Positivos',
    yaxis_title='Tasa de Verdaderos Positivos',
    height=450,
    template='plotly_white',
    legend=dict(x=0.6, y=0.1)
)
fig_roc.show()

In [17]:
# -- Curva de pérdida del entrenamiento ----------------------------------------------
fig_loss = go.Figure()
fig_loss.add_trace(go.Scatter(
    y=mlp.loss_curve_,
    mode='lines',
    name='Loss entrenamiento',
    line=dict(color='#8338EC', width=2.5)
))
fig_loss.add_trace(go.Scatter(
    y=mlp.validation_scores_,
    mode='lines',
    name='Score validación',
    line=dict(color='#FF006E', width=2.5, dash='dot')
))
fig_loss.update_layout(
    title='Curva de pérdida : MLP',
    xaxis_title='Iteración',
    yaxis_title='Valor',
    height=420,
    template='plotly_white',
    legend=dict(x=0.6, y=0.9)
)
fig_loss.show()

In [18]:
# -- Guardar resultados ----------------------------------------------
resultados_mlp = {
    'modelo'             : 'MLP',
    'accuracy'           : round(acc, 4),
    'f1_score'           : round(f1, 4),
    'roc_auc'            : round(auc, 4),
    'precision_fallecido': 0.30,
    'recall_fallecido'   : 0.64,
    'iteraciones'        : mlp.n_iter_,
    'loss_final'         : round(mlp.loss_, 4),
}

with open('../data/models/resultados_mlp.json', 'w') as f:
    json.dump(resultados_mlp, f, indent=2)

print('✓ Resultados guardados en data/models/resultados_mlp.json')
print(f'\nResumen MLP:')
for k, v in resultados_mlp.items():
    print(f'  {k:<25} {v}')

✓ Resultados guardados en data/models/resultados_mlp.json

Resumen MLP:
  modelo                    MLP
  accuracy                  0.65
  f1_score                  0.6861
  roc_auc                   0.6942
  precision_fallecido       0.3
  recall_fallecido          0.64
  iteraciones               57
  loss_final                0.5404


## 4. Resumen del modelo

| Métrica | Valor |
|---|---|
| Accuracy | 65.00% |
| F1-Score (weighted) | 68.61% |
| ROC-AUC | 69.42% |
| Precision Fallecido | 30% |
| Recall Fallecido | 64% |
| Iteraciones | 57 |
| Loss final | 0.5404 |

**Conclusiones:**
- Convergencia en 57 iteraciones con early stopping
- Loss desciende consistentemente : entrenamiento estable
- Score de validación se estabiliza en ~0.70 : sin sobreajuste